axes[0].imshow(img_2, cmap="gray")
axes[0].set_title("Lena Gray")
axes[1].imshow(quantized_3)
axes[1].set_title("6 bits")
axes[2].imshow(quantized_4)
axes[2].set_title("4 bits")
axes[3].imshow(quantized_5)
axes[3].set_title("2 bits")
axes[4].imshow(quantized_6)
axes[4].set_title("1 bit")

In [ ]:
ax.imshow(quantized.convert("RGB"))
    ax.set_title(f"{bits} bits/pixel\n(up to {color_count} colors)")

In [ ]:
median_array = np.median(noisy_stack, axis=0).astype(np.uint8)

final_median_img = Image.fromarray(median_array)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(brain, cmap="gray")
axes[0].set_title("Brain Original")

axes[1].imshow(final_median_img, cmap="gray")
axes[1].set_title("Brain Noise 32")

for ax in axes:
    ax.axis('off')
plt.show()

In [ ]:
images = {
    "Lena": lena,
    "Moon": moon,
    "Bay": bay,
    "Brain": brain,
}

colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)

for col, ((name, image), color) in enumerate(zip(images.items(), colors)):
    pixels = np.asarray(image)
    histogram, _ = np.histogram(pixels.ravel(), bins=256, range=(0, 256))
    intensities = np.arange(256)

    # Top row: original image
    axes[0, col].imshow(pixels, cmap="gray")
    axes[0, col].set_title(name)
    axes[0, col].axis("off")

    # Bottom row: histogram
    axes[1, col].fill_between(intensities, histogram, color=color, alpha=0.5)
    axes[1, col].plot(intensities, histogram, color=color)
    axes[1, col].set_title(f"{name} Histogram")
    axes[1, col].set_xlim(0, 255)
    axes[1, col].set_xlabel("Intensity (0-255)")
    axes[1, col].set_ylabel("Pixel count")

plt.show()

In [ ]:
images = {
    "Lena": lena,
    "Moon": moon,
    "Bay": bay,
    "Brain": brain,
}

for name, image in images.items():
    pixels = np.asarray(image)
    histogram, _ = np.histogram(
        pixels.ravel(),
        bins=256,
        range=(0, 256)
    )

    plt.figure()
    plt.plot(np.arange(256), histogram)
    plt.xlabel("Intensity (0-255)")
    plt.ylabel("Pixel count")
    plt.title(f"{name} Histogram")
    plt.xlim(0, 255)
    plt.show()

In [ ]:
from io import BytesIO
from urllib.request import urlopen
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

Lena = ("https://raw.githubusercontent.com/florencio300/""LISTA-1-PROC-SIN-II/main/dataset/lena2.tif")
Moon = ("https://raw.githubusercontent.com/florencio300/""LISTA-1-PROC-SIN-II/main/dataset/moon.jpg")
Bay = ("https://raw.githubusercontent.com/florencio300/""LISTA-1-PROC-SIN-II/main/dataset/bay.jpg")
Brain = ("https://raw.githubusercontent.com/florencio300/""LISTA-1-PROC-SIN-II/main/dataset/brain.jpg")

with urlopen(Lena) as response:
    lena = Image.open(BytesIO(response.read())) 
with urlopen(Moon) as response:
    moon = Image.open(BytesIO(response.read())) 
with urlopen(Bay) as response:
    bay = Image.open(BytesIO(response.read()))  
with urlopen(Brain) as response:
    brain = Image.open(BytesIO(response.read()))

images = {"Lena": lena,"Moon": moon,"Bay": bay,"Brain": brain,}

histograms = {}
equalized_histograms = {}
equalized_images = {}

fig, axes = plt.subplots(2, len(images), figsize=(4 * len(images), 8), squeeze=False)

for col, (name, image) in enumerate(images.items()):
    pixels = np.asarray(image, dtype=np.uint8)
 
    histogram, _ = np.histogram(pixels.ravel(), bins=256, range=(0, 256))

    pxl_ntst = histogram.cumsum()
    pxl_ntst_min = [pxl_ntst > 0][0]

    if pxl_ntst_min == pixels.size:  
        equalized_pixels = pixels.copy()
    else:
        lookup_table = np.rint((pxl_ntst - pxl_ntst_min) * 255 / (pixels.size - pxl_ntst_min)).clip(0, 255).astype(np.uint8)
        equalized_pixels = lookup_table[pixels]

    equalized_images[name] = Image.fromarray(equalized_pixels)

    axes[0, col].imshow(pixels, cmap="gray", vmin=0, vmax=255)
    axes[0, col].set_title(f"{name} — Original")
    axes[0, col].axis("off")

    axes[1, col].imshow(equalized_pixels, cmap="gray", vmin=0, vmax=255)
    axes[1, col].set_title(f"{name} — Equalized")
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2

histograms = {}
adaptive_equalized_images = {
    (8, 8): {},
    (16, 16): {},
}
equalized_histograms = {
    (8, 8): {},
    (16, 16): {},
}

tile_grids = [(8, 8), (16, 16)]

fig, axes = plt.subplots(
    3, len(images),
    figsize=(4 * len(images), 12),
    squeeze=False,
)

for col, (name, image) in enumerate(images.items()):
    # CLAHE expects a single-channel, 8-bit grayscale image.
    pixels = np.asarray(image.convert("L"), dtype=np.uint8)

    histograms[name], _ = np.histogram(
        pixels.ravel(), bins=256, range=(0, 256)
    )

    axes[0, col].imshow(pixels, cmap="gray", vmin=0, vmax=255)
    axes[0, col].set_title(f"{name} — Original")
    axes[0, col].axis("off")

    for row, tile_grid in enumerate(tile_grids, start=1):
        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=tile_grid,
        )
        equalized_pixels = clahe.apply(pixels)

        adaptive_equalized_images[tile_grid][name] = Image.fromarray(
            equalized_pixels
        )
        equalized_histograms[tile_grid][name], _ = np.histogram(
            equalized_pixels.ravel(), bins=256, range=(0, 256)
        )

        axes[row, col].imshow(
            equalized_pixels, cmap="gray", vmin=0, vmax=255
        )
        axes[row, col].set_title(
            f"{name} — CLAHE {tile_grid[0]}×{tile_grid[1]}"
        )
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
for row, tile_grid in enumerate(tile_grids, start=1):
        clahe = cv2.createCLAHE(clipLimit=2.0,tileGridSize=tile_grid,)
        adaptive_equalized_pixels = clahe.apply(pixels)

        adaptive_equalized_images[tile_grid][name] = Image.fromarray(adaptive_equalized_pixels)
        adaptive_equalized_histograms[tile_grid][name], _ = np.histogram(adaptive_equalized_pixels.ravel(), bins=256, range=(0, 256))

        axes[row, col].imshow(adaptive_equalized_pixels, cmap="gray", vmin=0, vmax=255)
        axes[row, col].set_title(f"{name} — CLAHE {tile_grid[0]}×{tile_grid[1]}")
        axes[row, col].axis("off")

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Localização do arquivo de imagem
image_url = ("https://raw.githubusercontent.com/florencio300/""LISTA-1-PROC-SIN-II/main/dataset/peppers3.tif")

# Abertura do arquivo de imagem
with urlopen(image_url) as response:
    original = Image.open(BytesIO(response.read())).convert("RGB")

image_array = np.array(original)

# Conversão de original para níveis de cinza
original_gray = original.convert("L")

# Obtenção dos canais individuais
R = image_array[:, :, 0]
G = image_array[:, :, 1]
B = image_array[:, :, 2]

# Binarização por limiar global para destacar o vermelho
# Critério simples: pixel vermelho tem R alto e G/B mais baixos
threshold = 150
mask = (R > threshold) & (R > G + 20) & (R > B + 20)

# Aplica a máscara na imagem original
result = np.copy(image_array)
result[~mask] = 0


fig, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)


# Plots e títulos
axes[0].imshow(original)
axes[0].set_title(f"Peppers Original")
axes[1].imshow(original_gray, cmap= "gray")
axes[1].set_title(f"Peppers Gray",)
axes[2].imshow(mask, cmap="gray")
axes[2].set_title(f"Peppers Mask")
axes[3].imshow(result, cmap="gray")
axes[3].set_title(f"Peppers Mask over Peppers Original")

for ax in axes:
    ax.axis("off")

In [ ]:
import numpy as np
from PIL import Image

# Carrega a imagem colorida
img = Image.open("Peppers_color.png").convert("RGB")
arr = np.array(img)

# 1) Converte para nível de cinza
gray = np.dot(arr[..., :3], [0.299, 0.587, 0.114]).astype(np.uint8)
gray_img = Image.fromarray(gray, mode="L")

# 2) Separa canais
R = arr[:, :, 0]
G = arr[:, :, 1]
B = arr[:, :, 2]

# 3) Binarização por limiar global para destacar o vermelho
# Critério simples: pixel vermelho tem R alto e G/B mais baixos
threshold = 150
mask = (R > threshold) & (R > G + 20) & (R > B + 20)

# 4) Aplica a máscara na imagem original
result = np.copy(arr)
result[~mask] = 0

# 5) Mostra os resultados
gray_img.show()
Image.fromarray(mask.astype(np.uint8) * 255, mode="L").show()
Image.fromarray(result, mode="RGB").show()